# Linear and Quadratic Programming in Python

This notebook demonstrates:

1. **Linear Programming (LP)** with OR-Tools `GLOP`
2. **Quadratic Programming (QP)** with SciPy `SLSQP`

The LP example follows the same structure as the OR-Tools style shown in your notes.

In [ ]:
# Numerical arrays and pretty printing
import numpy as np

# OR-Tools (for linear programming)
from ortools.linear_solver import pywraplp

# SciPy optimizer (for quadratic programming)
from scipy.optimize import minimize, LinearConstraint

# Make printed vectors easier to read
np.set_printoptions(precision=4, suppress=True)

## 1) Linear Programming Example (OR-Tools)

We solve:

- Maximize $z = 3x + 4y$
- Subject to:
  - $x + 2y \le 14$
  - $3x - y \ge 0$
  - $x - y \le 2$
  - $x, y \ge 0$

This matches the classic OR-Tools LP demo.

In [ ]:
def linear_programming_example():
    """
    Solve a Linear Program (LP) with OR-Tools.

    Current model:
        Maximize    z = 3x + 4y
        Subject to  x + 2y <= 14
                    3x - y >= 0
                    x - y <= 2
                    x, y >= 0

    HOW TO ADAPT THIS FOR A DIFFERENT LP:
    1) Change variable bounds in NumVar / IntVar lines.
    2) Change or add constraint lines (solver.Add(...)).
    3) Change objective expression in solver.Maximize(...) or solver.Minimize(...).
    4) If you need integer variables, use IntVar instead of NumVar and switch solver backend
       from "GLOP" to an integer-capable solver like "SAT" or "SCIP".
    """

    # For continuous LP, GLOP is a standard and fast choice.
    solver = pywraplp.Solver.CreateSolver("GLOP")
    if not solver:
        raise RuntimeError("GLOP solver is not available.")

    # ---------------------------
    # 1) Decision variables
    # ---------------------------
    # x, y are continuous and non-negative (lower bound = 0, upper bound = +infinity).
    # For a different problem, add more variables in the same way.
    x = solver.NumVar(0, solver.infinity(), "x")
    y = solver.NumVar(0, solver.infinity(), "y")

    print("Number of variables =", solver.NumVariables())

    # ---------------------------
    # 2) Constraints
    # ---------------------------
    # Each solver.Add(...) line represents one mathematical constraint.
    # Replace these expressions to match your own model.
    solver.Add(x + 2 * y <= 14.0)  # constraint 1
    solver.Add(3 * x - y >= 0.0)   # constraint 2
    solver.Add(x - y <= 2.0)       # constraint 3

    print("Number of constraints =", solver.NumConstraints())

    # ---------------------------
    # 3) Objective
    # ---------------------------
    # To solve a minimization LP, use solver.Minimize(...).
    # To change objective coefficients, edit numbers in this expression.
    solver.Maximize(3 * x + 4 * y)

    # ---------------------------
    # 4) Solve + Report
    # ---------------------------
    print(f"Solving with {solver.SolverVersion()}")
    status = solver.Solve()

    if status == pywraplp.Solver.OPTIMAL:
        print("Optimal objective value =", solver.Objective().Value())
        print("x =", x.solution_value())
        print("y =", y.solution_value())
        print("Wall time (ms) =", solver.wall_time())
        print("Iterations =", solver.iterations())
    else:
        print("The problem does not have an optimal solution.")


linear_programming_example()

Number of variables = 2
Number of constraints = 3
Solving with Glop solver v9.15.6755
Optimal objective value = 33.99999999999999
x = 5.999999999999998
y = 3.9999999999999996
Wall time (ms) = 4
Iterations = 2


## 2) Quadratic Programming Example (SciPy)

Now solve a quadratic program:

- Minimize $f(x, y) = x^2 + y^2 - 2x - 6y$
- Subject to:
  - $x + y \le 2$
  - $x \ge 0,
  \; y \ge 0$

This is a convex QP because the quadratic term matrix is positive semidefinite.

In [ ]:
def quadratic_programming_example():
    """
    Solve a convex Quadratic Program (QP) with SciPy SLSQP.

    We use this standard form:
        minimize    0.5 * v^T Q v + q^T v
        subject to  A v <= b
                    variable bounds

    Current settings produce the problem:
        minimize    x^2 + y^2 - 2x - 6y
        subject to  x + y <= 2
                    x >= 0, y >= 0

    HOW TO ADAPT THIS FOR A DIFFERENT QP:
    1) Replace Q and q with your own quadratic and linear objective terms.
    2) Replace A and b to represent your linear inequalities.
    3) Update bounds for each variable.
    4) Change initial guess x0 (often helps convergence speed).
    """

    # ------------------------------------------------------------
    # 1) Define objective in matrix form: 0.5*v^T*Q*v + q^T*v
    # ------------------------------------------------------------
    # For x^2 + y^2 - 2x - 6y:
    #   Q = [[2,0],[0,2]] and q = [-2,-6]
    # because 0.5*[x y]Q[x y]^T = x^2 + y^2.
    Q = np.array([
        [2.0, 0.0],
        [0.0, 2.0]
    ])
    q = np.array([-2.0, -6.0])

    def objective(v):
        return 0.5 * v @ Q @ v + q @ v

    def grad(v):
        # Gradient of 0.5*v^T*Q*v + q^T*v is Qv + q (when Q is symmetric).
        return Q @ v + q

    # ------------------------------------------------------------
    # 2) Linear inequality constraints: A v <= b
    # ------------------------------------------------------------
    # Current single inequality x + y <= 2:
    A = np.array([
        [1.0, 1.0]
    ])
    b = np.array([2.0])

    # SciPy LinearConstraint expects lower and upper bounds:
    # lb <= A v <= ub. For A v <= b, use lb = -inf and ub = b.
    lb = np.full(b.shape, -np.inf)
    ub = b
    linear_constraint = LinearConstraint(A, lb, ub)

    # ------------------------------------------------------------
    # 3) Variable bounds
    # ------------------------------------------------------------
    # (0, None) means variable >= 0 and unbounded above.
    # Add one tuple per variable, in the same order as v.
    bounds = [
        (0, None),  # x >= 0
        (0, None)   # y >= 0
    ]

    # ------------------------------------------------------------
    # 4) Initial guess + solve
    # ------------------------------------------------------------
    x0 = np.array([0.5, 0.5])

    result = minimize(
        objective,
        x0,
        method="SLSQP",
        jac=grad,
        bounds=bounds,
        constraints=[linear_constraint],
        options={"disp": False}
    )

    print("Success:", result.success)
    print("Message:", result.message)

    if result.success:
        x_opt, y_opt = result.x
        print(f"Optimal point: x = {x_opt:.4f}, y = {y_opt:.4f}")
        print(f"Minimum objective value: {result.fun:.4f}")
        print("Constraint check (x+y <= 2):", x_opt + y_opt)
    else:
        print("Optimization failed.")


quadratic_programming_example()

Success: True
Message: Optimization terminated successfully
Optimal point: x = 0.0000, y = 2.0000
Minimum objective value: -8.0000
Constraint check (x+y <= 2): 2.000000000000014
